# BME i9400 — Meeting 1
## Course Launch, Colab, and Biomedical Data Readiness

**Fall 2026 · Monday, August 31 · asynchronous**

I am at the Optios FAA meeting in DC today, so this first meeting is a recorded walkthrough of
this notebook. Watch the video with this notebook open beside it and run every cell as I do.

**What we cover today**

1. How this course works
2. Colab: getting a working Python environment in your browser
3. Data readiness I — **provenance**: where did this data come from, and who is in it?
4. Data readiness II — **label quality**: what does `y` actually mean?
5. **Splits and leakage** — the single most common way biomedical ML results turn out to be wrong
6. Your check-in deliverable

Budget about 45 minutes. Nothing here requires a GPU, and nothing takes more than a few seconds to run.

---
## Step 0 — Save your own copy first

**Do this before you run anything.**

`File ▸ Save a copy in Drive`

You are looking at a read-only course notebook. If you start typing in this one, your work will not
be saved. Once you have your own copy, the title bar will say *Copy of ...* — work in that one.

Two Colab habits worth building now:

- **`Shift + Enter`** runs the current cell and moves to the next one.
- If something gets into a weird state, `Runtime ▸ Restart session` and re-run from the top. This
  fixes the large majority of "it worked yesterday" problems.

---
## Part 1 — How this course works

The full syllabus is in the course repository. The parts that change what you do week to week:

**Two kinds of meetings.** *Concept* meetings are lectures with short Brightspace quizzes mixed in.
*Studio* meetings are twenty minutes of instruction and then forty-five minutes of notebook work in
the room.

**How you are graded.** In-class quizzes 10% · labs and defense quizzes 20% · midterm 20% ·
final concept exam 15% · capstone 35%.

**The one policy to understand now — generative AI.** You are *expected* to use AI on the three labs
and on the capstone. That is how you will work in your career. What gets graded is not the artifact
but whether you can defend it: each lab is followed by a ten-minute, closed-AI, in-class quiz, and
during that quiz you may consult **your own submitted notebook and nothing else**. Produce with AI;
then show me you understand what you produced.

**Two dates to put in your calendar right now.**

- **Meeting 11 is on Tuesday, October 13** — it follows a Monday schedule. The College is closed
  Monday the 12th.
- **The midterm is Wednesday, October 28.** It is before the November 6 withdrawal deadline, on
  purpose, so you have a real grade in hand while you can still act on it.

---
## Part 2 — Colab

Colab gives you a Python environment in the browser with the scientific stack already installed.
It is the required runtime for this course: there is nothing to install, and every student is running
the same thing, which means when something breaks I can actually reproduce it.

Run the cell below. It confirms you are in Colab and prints the versions of everything we will use.
Some of these numbers get recorded in your check-in at the end, so don't skip it.

In [ ]:
import sys, platform, importlib, importlib.util
from datetime import datetime, timezone

try:
    IN_COLAB = ("google.colab" in sys.modules
                or importlib.util.find_spec("google.colab") is not None)
except Exception:
    IN_COLAB = False   # find_spec raises if the parent package is absent

print("Running in Colab :", IN_COLAB)
print("Python           :", sys.version.split()[0])
print("Platform         :", platform.platform())
print()

ENV = {"in_colab": bool(IN_COLAB), "python": sys.version.split()[0]}
for pkg in ["numpy", "pandas", "matplotlib", "sklearn"]:
    try:
        m = importlib.import_module(pkg)
        ENV[pkg] = getattr(m, "__version__", "?")
    except Exception as e:
        ENV[pkg] = f"MISSING ({e.__class__.__name__})"
    print(f"{pkg:12s} : {ENV[pkg]}")

ENV["run_utc"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
print("\nTimestamp        :", ENV["run_utc"])

if not IN_COLAB:
    print("\nNOTE: you are not in Colab. That is allowed but unsupported — all graded work assumes Colab.")

---
## Part 3 — Data readiness I: provenance

Before you fit anything, you answer three questions. Not because it is good hygiene — because the
answers determine whether your model means anything.

1. **Where did this data come from?**
2. **Who is in it, and who is not?**
3. **How were the numbers actually produced?**

We will use the Pima Indians Diabetes dataset. It is one of the most-used teaching datasets in
machine learning, which makes it a good place to see how much can be wrong with a dataset that
everybody trusts.

In [ ]:
import pandas as pd

COURSE = "https://raw.githubusercontent.com/dmochow/BME-i9400-2026/main/data/diabetes.csv"
MIRROR = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
COLS = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin",
        "BMI","DiabetesPedigreeFunction","Age","Outcome"]

try:
    df = pd.read_csv(COURSE)
    print("loaded from the course repository")
except Exception:
    df = pd.read_csv(MIRROR, header=None, names=COLS)
    print("course URL unavailable — loaded from public mirror")

print(f"{df.shape[0]} rows x {df.shape[1]} columns\n")
df.head()

### Who is in this dataset?

Answering question 2 takes ten seconds of reading and changes everything downstream. This data comes
from a long-running study of Pima women, **age 21 and older**, from one community in Arizona. Every
row is a woman. Every row is from one population.

So a model trained here has a documented population of validity, and men, children, and other
ancestries are outside it. A model that scores well on this data is not a diabetes screening tool.
It is a diabetes screening tool *for the population that was sampled*. This is not a technicality —
it is the difference between a finding and a liability.

Run the cell below and look at `Age`.

In [ ]:
print(df[["Age","Pregnancies","BMI","Glucose"]].describe().round(1).to_string())
print("\nMinimum age in the dataset:", int(df.Age.min()))

### How were the numbers produced?

Question 3. Look for values that are *physically impossible* rather than merely unusual — they are
the fingerprint of a recording convention you have not been told about.

A living person cannot have a blood glucose of 0, a blood pressure of 0, or a BMI of 0. Those zeros
are not measurements. They are **missing values that someone stored as zero**, and no error is raised
when you load them.

In [ ]:
IMPOSSIBLE = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
zeros = (df[IMPOSSIBLE] == 0).sum()

print("Rows recording a physically impossible zero:\n")
for col, n in zeros.items():
    print(f"  {col:16s} {n:4d}   ({100*n/len(df):.1f}% of rows)")

n_affected = int((df[IMPOSSIBLE] == 0).any(axis=1).sum())
print(f"\nRows with at least one impossible zero: {n_affected} of {len(df)} "
      f"({100*n_affected/len(df):.1f}%)")
ENV["pima_zero_rows"] = n_affected

Roughly half the dataset. If you had loaded this file, called `.fit()`, and reported an accuracy —
which is exactly what thousands of tutorials do — you would have silently taught your model that
"insulin = 0" is a real physiological state describing half your patients.

**Nothing in the data told you this.** No exception, no warning. You found it by asking whether the
numbers were physically possible. That habit is most of what data readiness is.

---
## Part 4 — Data readiness II: label quality

Everyone scrutinizes `X`. Almost nobody scrutinizes `y`. Your model's ceiling is set by your label,
so a noisy or subtly mis-defined label caps everything you do afterward — and no amount of
architecture fixes it.

For every project in this course, including your capstone, you must be able to state:

- **What does `y = 1` literally mean?** Which criterion, applied by whom, using what instrument?
- **When was it measured** relative to the features?
- **How reliable is it?** Would two clinicians assign the same label?

Here, `Outcome = 1` means the patient met the WHO criterion for diabetes **within five years of the
examination that produced the features**. That is a *prognostic* label, not a diagnostic one — the
features come first, the label comes later. Get that backwards and you have built something that
predicts the past.

In [ ]:
rate = df.Outcome.mean()
print(f"Positive rate: {rate:.3f}   ({df.Outcome.sum()} of {len(df)} patients)")
print(f"\nA model that predicts 'no diabetes' for everyone would be {100*(1-rate):.1f}% accurate")
print("and would be completely useless. This is why accuracy is a poor metric for imbalanced")
print("clinical problems, and why we will spend Meetings 8-10 on what to use instead.")

---
## Part 5 — Splits and leakage

This is the part to actually pay attention to.

**Leakage** is when information reaches your model during training that would not be available at
prediction time in the real world. It does not produce an error. It produces a *great result* — which
is what makes it dangerous. Most biomedical ML findings that fail to replicate failed here.

The most common form in biomedical data is the simplest: **the same subject appears in both your
training set and your test set.** Biomedical datasets are almost never one row per person. They are
repeated visits, multiple images per patient, many windows from one recording, several slides per
tumor. If you split those rows at random, the same patient lands on both sides, and your model gets
to recognize the *person* instead of the *disease*.

Let's build exactly that situation and measure what it costs. Below: 100 patients, each with 4–8
visits. The diagnosis is a property of the patient, and there is a real but weak biological signal
in the features.

In [ ]:
import numpy as np

SEED = 9400
rng = np.random.default_rng(SEED)
N_PATIENTS, N_FEAT = 100, 12

# Each patient has a stable physiological "fingerprint" that persists across visits.
fingerprint = rng.normal(0, 1.0, size=(N_PATIENTS, N_FEAT))

# The real biological signal: risk depends on only the first three features, and is noisy.
w = np.zeros(N_FEAT); w[:3] = [1.1, -0.9, 0.8]
risk  = fingerprint @ w + rng.normal(0, 1.6, size=N_PATIENTS)
label = (risk > np.median(risk)).astype(int)          # diagnosis is a property of the PATIENT

n_visits = rng.integers(4, 9, size=N_PATIENTS)
X, patient_id, y = [], [], []
for i in range(N_PATIENTS):
    for _ in range(n_visits[i]):
        X.append(fingerprint[i] + rng.normal(0, 0.18, size=N_FEAT))   # small visit-to-visit noise
        patient_id.append(i)
        y.append(label[i])

X, patient_id, y = np.array(X), np.array(patient_id), np.array(y)
print(f"{X.shape[0]} visits from {N_PATIENTS} patients")
print(f"{X.shape[1]} features per visit, positive rate {y.mean():.2f}")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def evaluate(train_idx, test_idx):
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
    model.fit(X[train_idx], y[train_idx])
    return roc_auc_score(y[test_idx], model.predict_proba(X[test_idx])[:, 1])

# --- Split 1: randomly, by visit. The tempting default. ---
tr, te = train_test_split(np.arange(len(y)), test_size=0.3, random_state=SEED, stratify=y)
auc_leaky  = evaluate(tr, te)
overlap    = len(set(patient_id[tr]) & set(patient_id[te]))

# --- Split 2: by patient. No one appears on both sides. ---
tr2, te2 = next(GroupShuffleSplit(n_splits=1, test_size=0.3,
                                  random_state=SEED).split(X, y, groups=patient_id))
auc_honest = evaluate(tr2, te2)

print(f"Random split by VISIT     AUC = {auc_leaky:.3f}   "
      f"patients appearing on BOTH sides: {overlap}")
print(f"Grouped split by PATIENT  AUC = {auc_honest:.3f}   "
      f"patients appearing on both sides: {len(set(patient_id[tr2]) & set(patient_id[te2]))}")
print(f"\nCost of the mistake: {auc_leaky - auc_honest:.3f} AUC")

ENV["auc_leaky"]  = round(float(auc_leaky), 3)
ENV["auc_honest"] = round(float(auc_honest), 3)

### What just happened

The first number is nearly perfect. It is also entirely fake.

With a random split, 89 of the 100 patients have some visits in training and other visits in test.
The model is a nearest-neighbour classifier, so for a test visit it finds the closest training
example — which is *that same patient's visit from a different month*, sitting a fraction of a
standard deviation away. It copies that patient's label. It never learned anything about disease.
It learned to recognize people.

The second number is what the model is actually worth on a patient it has never seen. That is the
only number you would be allowed to publish, and it is the number the model would deliver in a clinic.

A result of 0.998 should never make you happy. It should make you suspicious.

### Three ways to leak, all of which we will hit again

| | What it looks like | The fix |
|---|---|---|
| **Group leakage** | Same patient, slide, or recording on both sides of the split | Split by the *group*, not the row — `GroupShuffleSplit`, `GroupKFold` |
| **Preprocessing leakage** | Scaling, imputing, or feature-selecting using statistics computed over *all* the data | Fit every transform on train only — use a `Pipeline`, as above |
| **Temporal leakage** | A feature that is only recorded *after* the outcome is known (treatment started, discharge code, length of stay) | Ask of every feature: was this knowable at prediction time? |

We will make all three of these concrete in Meetings 11 and 12. For now, the habit worth forming
today: **before you split, ask what a row is, and ask whether rows repeat.**

---
## Part 6 — Your check-in

This is your deliverable for Meeting 1. It is short, and it is due **before Meeting 2 on Wednesday,
September 2**.

Fill in the two cells below and run them. The second cell will print a small file for you to submit.

In [ ]:
# ---- Fill these in -------------------------------------------------------
NAME            = ""      # e.g. "Ada Lovelace"
EMAIL           = ""      # your @citymail.cuny.edu address
GITHUB_USERNAME = ""      # your GitHub username; make a free account if you do not have one
# --------------------------------------------------------------------------

assert NAME and EMAIL and GITHUB_USERNAME, "Fill in all three fields above, then re-run this cell."
print(f"{NAME} <{EMAIL}>  ·  github.com/{GITHUB_USERNAME}")

In [ ]:
# ---- Four short answers. One or two sentences each. No AI on this cell. ----

# 1. The random-by-visit split scored 0.998 and the by-patient split scored 0.601.
#    In your own words, why?
A1 = ""

# 2. Name one group of people this Pima-trained model should NOT be used on, and say why.
A2 = ""

# 3. Think about data you have worked with, or expect to work with in your own research.
#    Describe one specific way it could leak.
A3 = ""

# 4. One question you want answered in this course. Anything. This one is not graded.
A4 = ""
# ---------------------------------------------------------------------------

for i, a in enumerate([A1, A2, A3], start=1):
    assert len(a.strip()) >= 20, f"Answer {i} looks empty or very short — give it a real sentence."
print("Answers recorded.")

In [ ]:
lines = [
    f"# Meeting 1 check-in — {NAME}",
    "",
    f"- **Email:** {EMAIL}",
    f"- **GitHub:** @{GITHUB_USERNAME}",
    f"- **Notebook run:** {ENV['run_utc']}",
    "",
    "## Environment",
    "",
    f"- Colab: `{ENV['in_colab']}` · Python `{ENV['python']}`",
    f"- numpy `{ENV['numpy']}` · pandas `{ENV['pandas']}` · "
    f"matplotlib `{ENV['matplotlib']}` · scikit-learn `{ENV['sklearn']}`",
    "",
    "## Results I reproduced",
    "",
    f"- Pima rows with at least one impossible zero: **{ENV['pima_zero_rows']}** of 768",
    f"- AUC, random split by visit: **{ENV['auc_leaky']}**",
    f"- AUC, grouped split by patient: **{ENV['auc_honest']}**",
    "",
    "## Answers",
    "",
    f"**1. Why the two AUCs differ.** {A1.strip()}",
    "",
    f"**2. Who this model should not be used on.** {A2.strip()}",
    "",
    f"**3. A leakage risk in data I work with.** {A3.strip()}",
    "",
    f"**4. A question for the course.** {A4.strip() or '(none)'}",
    "",
]
CHECKIN = "\n".join(lines)

print("=" * 78)
print(CHECKIN)
print("=" * 78)
print(f"\nCopy everything between the lines above.")
print(f"Your filename is:  checkins/{GITHUB_USERNAME}.md")
print(f"\nSubmit here:")
print(f"  https://github.com/dmochow/BME-i9400-2026/new/main?filename=checkins/{GITHUB_USERNAME}.md")

### How to submit

The link printed above opens GitHub's file editor, already pointed at the right filename.

1. **Click the link.** Sign in to GitHub (create a free account first if you need one).
2. GitHub will tell you that you do not have write access and that it will **create a fork** —
   that is expected and correct. Let it.
3. **Paste** your check-in text into the editor.
4. Click **Commit changes**, then **Propose changes**, then **Create pull request**.

That is it. You have just opened your first pull request, which is how essentially all collaborative
scientific code gets reviewed. I will see all of them in one place.

**If GitHub gives you trouble**, do not spend more than fifteen minutes on it. Download this notebook
(`File ▸ Download ▸ .ipynb`) and upload it to the Meeting 1 check-in assignment on Brightspace
instead, and send me a one-line note saying GitHub blocked you. You will get full credit either way.

---

### Before Wednesday

- Submit the check-in above.
- Read the syllabus in full — it is three pages, and the AI policy in particular will affect how you
  work all semester.
- Bring a laptop to Meeting 2.

See you Wednesday. — JD